# EA1 — Carga y creación de dataset

**Objetivo:** obtener un dataset desde Hugging Face, transformarlo y generar un resumen en `.txt`.

Flujo:

```
HFDatasetLoader → DataFrame → transformaciones → DatasetSummarizer → output/resumen_iris.txt
```

- `src/hf_dataset_loader.py` → clase **HFDatasetLoader** (conexión y carga desde Hugging Face).
- `src/dataset_summarizer.py` → clase **DatasetSummarizer** (resumen y exportación a `.txt`).
- Este notebook solo **coordina** las clases y realiza las **transformaciones**.

## 1. Importar las clases desde sus archivos `.py`

In [1]:
import sys
from pathlib import Path

# Agrega la carpeta raíz del proyecto al path para poder importar desde src/
RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(RAIZ))

# Oculta advertencias informativas de las librerías (no afectan el resultado)
import warnings
warnings.filterwarnings("ignore")
from huggingface_hub.utils import logging as hf_logging
hf_logging.set_verbosity_error()

import pandas as pd
from src.hf_dataset_loader import HFDatasetLoader
from src.dataset_summarizer import DatasetSummarizer

print("Clases importadas correctamente:", HFDatasetLoader.__name__, DatasetSummarizer.__name__)

Clases importadas correctamente: HFDatasetLoader DatasetSummarizer


## 2. Obtener el DataFrame desde Hugging Face

Se usa el dataset `scikit-learn/iris`, split `train`.

In [2]:
loader = HFDatasetLoader("scikit-learn/iris", split="train")
print("Splits disponibles:", loader.available_splits())

df_original = loader.load()
print(loader.info())
df_original.head()

Splits disponibles: ['train']


{'dataset_id': 'scikit-learn/iris', 'split': 'train', 'filas': 150, 'columnas': 6}


,Id,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,1,5.1,3.5,1.4,0.2,Iris-setosa
1,2,4.9,3.0,1.4,0.2,Iris-setosa
2,3,4.7,3.2,1.3,0.2,Iris-setosa
3,4,4.6,3.1,1.5,0.2,Iris-setosa
4,5,5.0,3.6,1.4,0.2,Iris-setosa


In [3]:
df_original.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             150 non-null    int64  
 1   SepalLengthCm  150 non-null    float64
 2   SepalWidthCm   150 non-null    float64
 3   PetalLengthCm  150 non-null    float64
 4   PetalWidthCm   150 non-null    float64
 5   Species        150 non-null    object 
dtypes: float64(4), int64(1), object(1)
memory usage: 7.2+ KB


## 3. Transformaciones del DataFrame

Cada transformación genera un **nuevo DataFrame** que se guarda en una variable y se usa en el paso siguiente.

### Transformación 1 — Seleccionar columnas (eliminar `Id`)

La columna `Id` es solo un consecutivo y no aporta información para el análisis.

In [4]:
df_t1 = df_original.drop(columns=["Id"])
print("Columnas antes:", list(df_original.columns))
print("Columnas después:", list(df_t1.columns))
df_t1.head()

Columnas antes: ['Id', 'SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']
Columnas después: ['SepalLengthCm', 'SepalWidthCm', 'PetalLengthCm', 'PetalWidthCm', 'Species']


,SepalLengthCm,SepalWidthCm,PetalLengthCm,PetalWidthCm,Species
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### Transformación 2 — Renombrar columnas

Se pasan los nombres a español y en formato `snake_case`.

In [5]:
nombres = {
    "SepalLengthCm": "largo_sepalo_cm",
    "SepalWidthCm": "ancho_sepalo_cm",
    "PetalLengthCm": "largo_petalo_cm",
    "PetalWidthCm": "ancho_petalo_cm",
    "Species": "especie",
}
df_t2 = df_t1.rename(columns=nombres)
df_t2.head()

,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


### Transformación 3 — Limpiar valores de texto

Se quita el prefijo `Iris-` de la especie y se convierte la columna al tipo `category`.

In [6]:
df_t3 = df_t2.copy()
df_t3["especie"] = df_t3["especie"].str.replace("Iris-", "", regex=False).astype("category")
print("Antes:", df_t2["especie"].unique().tolist())
print("Después:", df_t3["especie"].cat.categories.tolist())
print("Tipo:", df_t3["especie"].dtype)

Antes: ['Iris-setosa', 'Iris-versicolor', 'Iris-virginica']
Después: ['setosa', 'versicolor', 'virginica']
Tipo: category


### Transformación 4 — Crear una columna calculada

`area_petalo_cm2` = largo × ancho del pétalo (aproximación del área).

In [7]:
df_t4 = df_t3.assign(area_petalo_cm2=(df_t3["largo_petalo_cm"] * df_t3["ancho_petalo_cm"]).round(2))
df_t4.head()

,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,area_petalo_cm2
0,5.1,3.5,1.4,0.2,setosa,0.28
1,4.9,3.0,1.4,0.2,setosa,0.28
2,4.7,3.2,1.3,0.2,setosa,0.26
3,4.6,3.1,1.5,0.2,setosa,0.30
4,5.0,3.6,1.4,0.2,setosa,0.28


### Transformación 5 — Filtrar filas con una condición

Se conservan solo las flores con largo de sépalo mayor a 5 cm.

In [8]:
df_transformado = df_t4[df_t4["largo_sepalo_cm"] > 5].reset_index(drop=True)
print("Filas antes del filtro:", len(df_t4))
print("Filas después del filtro:", len(df_transformado))
df_transformado.head()

Filas antes del filtro: 150
Filas después del filtro: 118


,largo_sepalo_cm,ancho_sepalo_cm,largo_petalo_cm,ancho_petalo_cm,especie,area_petalo_cm2
0,5.1,3.5,1.4,0.2,setosa,0.28
1,5.4,3.9,1.7,0.4,setosa,0.68
2,5.4,3.7,1.5,0.2,setosa,0.30
3,5.8,4.0,1.2,0.2,setosa,0.24
4,5.7,4.4,1.5,0.4,setosa,0.60


### Comparación: DataFrame original vs. transformado

In [9]:
comparacion = pd.DataFrame({
    "original": [df_original.shape[0], df_original.shape[1], ", ".join(df_original.columns)],
    "transformado": [df_transformado.shape[0], df_transformado.shape[1], ", ".join(df_transformado.columns)],
}, index=["filas", "columnas", "nombres de columnas"])
comparacion

,original,transformado
filas,150,118
columnas,6,6
nombres de columnas,"Id, SepalLengthCm, SepalWidthCm, PetalLengthCm...","largo_sepalo_cm, ancho_sepalo_cm, largo_petalo..."


In [10]:
df_transformado.groupby("especie", observed=True).size().rename("cantidad")

especie
setosa        22
versicolor    47
virginica     49
Name: cantidad, dtype: int64

## 4. Generar el resumen `.txt` con la segunda clase

Se pasa el **DataFrame transformado** (`df_transformado`) a `DatasetSummarizer`.

In [11]:
summarizer = DatasetSummarizer(df_transformado, nombre="scikit-learn/iris (transformado)")
ruta_resumen = summarizer.save(RAIZ / "output" / "resumen_iris.txt")
print("Resumen guardado en:", ruta_resumen.relative_to(RAIZ))

Resumen guardado en:

 output\resumen_iris.txt


## 5. Mostrar el resultado

In [12]:
print(ruta_resumen.read_text(encoding="utf-8"))

RESUMEN DEL DATASET: scikit-learn/iris (transformado)
Generado: 2026-10-14 17:30:33

1. DIMENSIONES
----------------------------------------------------------------------
Filas: 118
Columnas: 6

2. COLUMNAS: TIPOS, NULOS Y VALORES ÚNICOS
----------------------------------------------------------------------
                     tipo  nulos  % nulos  valores_unicos
columna                                                  
largo_sepalo_cm   float64      0      0.0              27
ancho_sepalo_cm   float64      0      0.0              22
largo_petalo_cm   float64      0      0.0              40
ancho_petalo_cm   float64      0      0.0              21
especie          category      0      0.0               3
area_petalo_cm2   float64      0      0.0              90

3. VALORES NULOS
----------------------------------------------------------------------
No hay valores nulos en el DataFrame.

4. ESTADÍSTICAS DESCRIPTIVAS (NUMÉRICAS)
----------------------------------------------------------

## 6. Explicación de las transformaciones

| # | Transformación | Operación de pandas | Resultado |
|---|---|---|---|
| 1 | Seleccionar columnas | `drop(columns=["Id"])` | Se elimina el consecutivo `Id` (6 → 5 columnas). |
| 2 | Renombrar columnas | `rename(columns=...)` | Nombres en español y `snake_case`. |
| 3 | Limpiar texto y cambiar tipo | `str.replace(...)` + `astype("category")` | `Iris-setosa` → `setosa`; la especie pasa a tipo `category`. |
| 4 | Columna calculada | `assign(area_petalo_cm2=...)` | Nueva columna con el área aproximada del pétalo. |
| 5 | Filtrar filas | `df[df["largo_sepalo_cm"] > 5]` | Solo flores con sépalo > 5 cm (150 → 118 filas). |

El archivo `output/resumen_iris.txt` se generó a partir de `df_transformado`, por lo que corresponde a los datos **ya transformados**.